# 02 — Causal inference: estimate the effect

The first two notebooks prepared the data and explored the observed associations. Now answer the causal question:

> **To what extent does applying the backdoor defense, instead of baseline random filtering, change the probability of successful backdoor detection?**

| Variable | Meaning |
|---|---|
| `treatment = 0` | random filtering baseline |
| `treatment = 1` | backdoor defense applied |
| `outcome = 0` | detection failed |
| `outcome = 1` | detection succeeded |

You will work in four steps: **build the DAG → identify the estimand → estimate the ATE → run refutation checks.**

**Tutorial path:** 00 Data preparation → 01 Correlational analysis → **02 Causal inference**


## 1. The causal estimand

For each unit, imagine two potential outcomes:

- **Y(1)** — would detection succeed if the backdoor defense were applied?
- **Y(0)** — would detection succeed under random filtering?

Only one of the two is ever observed for a given unit, so the tutorial targets the **Average Treatment Effect**:

**ATE = E[Y(1) − Y(0)]**

The outcome is binary, so the ATE is a difference in detection-success probability. An ATE of `0.10` means the defense causes an estimated **10 percentage-point increase** in the detection success rate, on average, compared with random filtering.


## 2. Configure the causal analysis

Point the notebook at the dataset from notebook 00 and the worksheet from notebook 01.

The ground-truth paths are listed here too, but nothing reads them until the reveal section — after you have frozen your DAG.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display
from dowhy import CausalModel

from src.causal_graph_ui import CausalGraphBuilder
from src.causal_tutorial_utils import (
    check_dowhy_version,
    load_analysis_data,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

def default_params():
    return {
        "causal_dataset": "data/causal_data.csv",
        "dag_worksheet_path": "data/dag_worksheet.csv",
        "ground_truth_dag": "data/synthetic_ground_truth_edges.csv",
        "study_metadata": "data/synthetic_study_metadata.json",
        "treatment_column": "treatment",
        "outcome_column": "outcome",
        "covariate_columns": ['code_number_tokens', 'code_complexity', 'code_num_identifiers', 'code_num_strings', 'reviewer_experience', 'rollout_eligibility', 'noise_feature', 'inspection_intensity', 'manual_review_flag'],
        "graph_palette": "colorblind",
        "graph_edge_opacity": 0.35,
        "refuter_simulations": 50,
    }

params = default_params()

print("DoWhy version:", check_dowhy_version("0.14"))
params


## 3. Load the observed data

The model sees only the observed treatment, outcome, and measured variables. The generating DAG stays hidden.


In [ ]:
(
    analysis_df,
    treatment,
    outcome,
    covariates,
    excluded_covariates,
) = load_analysis_data(params)

print(f"Rows: {len(analysis_df):,}")
print(f"Backdoor-defense prevalence: {analysis_df[treatment].mean():.3f}")
print(f"Observed detection success rate: {analysis_df[outcome].mean():.3f}")
print(f"Covariates available for the graph: {len(covariates)}")
print(f"Excluded constant covariates: {excluded_covariates or 'none'}")

analysis_df.head()


## 4. Review your DAG worksheet

The worksheet records what the data showed, when each variable was measured, and the causal hypotheses you wrote down.

The worksheet is not itself a causal model. The DAG you build next is where the assumptions become explicit.


In [ ]:
worksheet_path = Path(params["dag_worksheet_path"])

if worksheet_path.exists():
    dag_worksheet = pd.read_csv(worksheet_path)
    display(dag_worksheet)
else:
    dag_worksheet = None
    print(
        "DAG worksheet not found. Run "
        "01_correlational_analysis.ipynb first."
    )


## 5. Build the DAG

The graph starts with the relationship under study:

`treatment → outcome`

That arrow is the hypothesis that applying the defense changes detection success.

Now add the other variables. **Add an edge only when you can state a plausible mechanism and a consistent time order.** Keep in mind:

- a pre-treatment variable associated with both treatment and outcome may be a common cause;
- a post-treatment variable may sit on the causal pathway;
- a variable caused by two others may be a collider;
- a variable can predict treatment strongly without causing the outcome at all.

The editor labels structural roles **after** you draw the graph. It does not infer arrows from correlation.


In [ ]:
graph_builder = CausalGraphBuilder(
    data_columns=analysis_df.columns,
    treatment=treatment,
    outcome=outcome,
    covariates=covariates,
    palette=params["graph_palette"],
    edge_opacity=params["graph_edge_opacity"],
)

graph_builder.display()


### Freeze your DAG

When the graph shows assumptions you are willing to defend, click **Use this DAG for analysis**, then run the next cell.

Do this before revealing the ground truth.


In [ ]:
analysis_dag = graph_builder.get_frozen_graph()

print(
    f"Using DAG with {analysis_dag.number_of_nodes()} nodes "
    f"and {analysis_dag.number_of_edges()} edges."
)


## 6. Create the DoWhy causal model

DoWhy combines the data, the treatment, the outcome, and the DAG you supplied.

The DAG — not the correlation matrix — determines which assumptions DoWhy works from.


In [ ]:
causal_model = CausalModel(
    data=analysis_df,
    treatment=treatment,
    outcome=outcome,
    graph=analysis_dag,
)

print("DoWhy causal model created from the frozen DAG.")


## 7. Identify the causal effect

Identification asks:

> **If this DAG is correct, can the ATE be written in terms of quantities we can observe?**

This step settles **what** to estimate, before you choose **how** to estimate it.


In [ ]:
identified_estimand = causal_model.identify_effect(
    proceed_when_unidentifiable=False
)

print(identified_estimand)


## 8. Estimate the ATE

Estimate the identified backdoor estimand using inverse propensity-score weighting.

Read the result in context:

- **positive** — the defense increases detection success on average;
- **negative** — the defense decreases it;
- **near zero** — little average change, under your model's assumptions.


In [ ]:
estimate = causal_model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_weighting",
    target_units="ate",
    control_value=0,
    treatment_value=1,
    method_params={
        "min_ps_score": 0.05,
        "max_ps_score": 0.95,
        "weighting_scheme": "ips_weight",
    },
)

estimated_ate = float(estimate.value)

print(estimate)
print(f"\nEstimated ATE: {estimated_ate:.4f}")
print(
    "Interpretation: the model estimates a "
    f"{estimated_ate * 100:.1f} percentage-point average change "
    "in detection success when using the backdoor defense rather than "
    "random filtering."
)


## 9. Refute the estimate

Refuters check whether the estimate behaves sensibly when the data is perturbed. They are diagnostics: they cannot prove the DAG is correct or that all confounding has been removed.

### Placebo treatment

Treatment is randomly permuted, which destroys the real assignment. The placebo effect should come out near zero — if it does not, something is wrong with the model.


In [ ]:
placebo_refutation = causal_model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
    num_simulations=params["refuter_simulations"],
    random_seed=RANDOM_SEED,
)

print(placebo_refutation)


### Random common cause

An irrelevant random variable is added to the model. A stable estimate should barely move.


In [ ]:
random_common_cause_refutation = causal_model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="random_common_cause",
    num_simulations=params["refuter_simulations"],
    random_seed=RANDOM_SEED,
)

print(random_common_cause_refutation)


# Reveal: compare with the synthetic ground truth

Only now open the files that stayed hidden while you built the graph.

Compare:

1. the DAG you proposed against the DAG that generated the data;
2. your estimated ATE against the known synthetic ATE.

Do not just count matching edges. For every edge you missed or added, ask **why** the correlational evidence pointed the way it did.


In [ ]:
ground_truth = pd.read_csv(params["ground_truth_dag"])
display(ground_truth)

true_edges = set(zip(ground_truth["source"], ground_truth["target"]))
proposed_edges = set(analysis_dag.edges())

missing_edges = sorted(true_edges - proposed_edges)
extra_edges = sorted(proposed_edges - true_edges)

print("\nEdges in the synthetic DAG but missing from your DAG:")
print(missing_edges or "none")

print("\nEdges in your DAG but not in the synthetic DAG:")
print(extra_edges or "none")

metadata_path = Path(params["study_metadata"])
if metadata_path.exists():
    study_metadata = json.loads(metadata_path.read_text())
    true_ate = float(study_metadata["true_average_treatment_effect"])
    print(f"\nKnown synthetic ATE: {true_ate:.4f}")
    print(f"Estimated ATE:       {estimated_ate:.4f}")
    print(f"Estimation error:    {estimated_ate - true_ate:+.4f}")
else:
    print("\nSynthetic study metadata not found; rerun notebook 00.")


## Final interpretation

The three notebooks answered three different questions:

| Notebook | Question | Answer |
|---|---|---|
| 00 | What was observed? | One record per unit, with one treatment and one outcome. |
| 01 | What patterns are visible? | Associations and imbalances, not causal effects. |
| 02 | What is the causal effect? | A DAG, an identified estimand, an estimated ATE, and robustness checks. |

The lesson: choosing which variables to adjust for is a **causal reasoning problem**, not a correlation-ranking exercise.
